In [ ]:
#| default_exp core

In [ ]:
#| export
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional, Union
import logging

In [ ]:
#| export
def validate_nside(nside: int) -> int:
    """Validate that nside is a power of 2.
    
    Args:
        nside: HEALPix resolution parameter
        
    Returns:
        Validated nside value
        
    Raises:
        ValueError: If nside is not a power of 2
        
    Examples:
        >>> validate_nside(64)
        64
        >>> validate_nside(100)
        Traceback (most recent call last):
        ...
        ValueError: nside must be a power of 2, got 100
    """
    if nside <= 0 or (nside & (nside - 1)) != 0:
        raise ValueError(f"nside must be a power of 2, got {nside}")
    return nside

In [ ]:
#| export
def mad(arr: np.ndarray) -> float:
    """Compute Median Absolute Deviation.
    
    Args:
        arr: Input array
        
    Returns:
        MAD value (float)
        
    Examples:
        >>> arr = np.array([1, 2, 3, 4, 5])
        >>> mad(arr)
        1.0
    """
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan")
    return float(np.median(np.abs(arr - np.median(arr))))

In [ ]:
#| export
def robust_std(arr: np.ndarray) -> float:
    """Compute robust standard deviation using MAD * 1.4826.
    
    The factor 1.4826 makes MAD consistent with standard deviation
    for normally distributed data.
    
    Args:
        arr: Input array
        
    Returns:
        Robust standard deviation (float)
        
    Examples:
        >>> arr = np.array([1, 2, 3, 4, 5])
        >>> robust_std(arr)
        1.4826
    """
    return mad(arr) * 1.4826

In [ ]:
#| export
def setup_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    """Setup a logger with standard formatting.
    
    Args:
        name: Logger name
        level: Logging level (default: INFO)
        
    Returns:
        Configured logger
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)
    
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(
            logging.Formatter('%(asctime)s %(levelname)s %(message)s')
        )
        logger.addHandler(handler)
    
    return logger

## Tests

In [ ]:
# Test validate_nside
assert validate_nside(64) == 64
assert validate_nside(128) == 128

try:
    validate_nside(100)
    assert False, "Should have raised ValueError"
except ValueError:
    pass

In [ ]:
# Test MAD
arr = np.array([1, 2, 3, 4, 5])
assert mad(arr) == 1.0

# Test with NaNs
arr_nan = np.array([1, 2, np.nan, 4, 5])
assert mad(arr_nan) == 1.0

In [ ]:
# Test robust_std
arr = np.array([1, 2, 3, 4, 5])
assert abs(robust_std(arr) - 1.4826) < 0.001